In [315]:

from urllib import request, error
from typing import List, Dict, Any, Iterable, Tuple, Optional

from pathlib import Path
import json, random, itertools, os, gc, statistics as st
import seqeval
import openai
import re
import requests

import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import KFold, train_test_split
from collections import Counter
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import BallTree
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity, pairwise_distances
from sklearn.cluster import KMeans
from dotenv import load_dotenv
from IPython.display import clear_output

import torch
from transformers import (
    AutoTokenizer,
        AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

from seqeval.metrics import classification_report, f1_score, precision_score, recall_score, accuracy_score
from seqeval.scheme import IOB2
from evaluate import load as load_metric
from tqdm.auto import tqdm

In [2]:
HOST = "http://127.0.0.1:11434"

# >>> Escolha um que REALMENTE está instalado (copie/cole da sua /api/tags):
MODEL_KEY = "deepseek-r1:14b"
# Alternativas que você tem: "qwen2.5:14b-instruct", "deepseek-r1:14b"

# Catálogo só para opções (não interfere no nome/tag do modelo)
OLLAMA_MODELS = {
    "qwen2.5:14b-instruct": {"options": {"num_ctx": 12000, "temperature": 0.2}},
    "llama3.1:8b":          {"options": {"num_ctx": 8000,  "temperature": 0.2}},
    "deepseek-r1:14b":      {"options": {"num_ctx": 8000,  "temperature": 0.2}},
}

In [3]:
OLLAMA_MODELS[MODEL_KEY]["options"].update({
    "temperature": 0.0,
    "repeat_penalty": 1.1,
})

In [4]:
JSON_PATH = "../data/geocorpus-v2.json"        # ajuste se estiver noutra pasta
SEED_GLOBAL = 42
FEW_SHOT_K = 20

random.seed(SEED_GLOBAL)
np.random.seed(SEED_GLOBAL)

# ---------- ler o arquivo ----------
with open(JSON_PATH, encoding="utf-8") as f:
    raw = json.load(f)



In [5]:
records_geo = [
    {
        "sentence_id": i,
        "tokens"     : item["tokens"],
        "ner_tags"   : item["ner_tokens"],
    }
    for i, item in enumerate(raw)
]

geocorpus_full = Dataset.from_list(records_geo)

In [6]:
# lista de rótulos (ordem alfabética garante consistência entre runs)
label_list = sorted({l for sent in geocorpus_full["ner_tags"] for l in sent})
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for l, i in label2id.items()}
NUM_LABELS = len(label_list)

In [7]:
iob_labels = (
    "O "
    "B-baciaSedimentar I-baciaSedimentar "
    "B-epoca I-epoca "
    "B-idade I-idade "
    "B-periodo I-periodo "
    "B-eon I-eon "
    "B-era I-era "
    "B-magmaticas I-magmaticas "
    "B-metamorficas I-metamorficas "
    "B-sedimentaresSiliciclasticas I-sedimentaresSiliciclasticas "
    "B-sedimentaresCarbonaticas I-sedimentaresCarbonaticas "
    "B-unidadeEstratigrafica I-unidadeEstratigrafica "
    "B-contextoGeologicoDeBacia I-contextoGeologicoDeBacia "
    "B-ambienteSedimentacao I-ambienteSedimentacao "
    "B-constituinteRochaSedimentar I-constituinteRochaSedimentar "
    "B-fosseis I-fosseis "
    "B-planctonico I-planctonico "
    "B-bentonico I-bentonico "
    "B-mineral I-mineral "
    "B-procedimentoMetodologico I-procedimentoMetodologico"
)

# Splits

In [8]:
def random_splits(
    ds: Dataset, test_size=0.2, seeds: List[int] = range(30)
) -> List[DatasetDict]:
    triples = []
    for s in seeds:
        train, dev = ds.train_test_split(test_size=test_size, seed=s).values()
        triples.append(DatasetDict(train=train, dev=dev))
    return triples

In [9]:
# def split_heur_length(ds: Dataset, top_pct: float = 0.20) -> DatasetDict:
#     lengths = np.array([len(t) for t in ds["tokens"]])
#     thr = np.percentile(lengths, 100 * (1 - top_pct))
#     mask = lengths >= thr
#     return DatasetDict(train=ds.filter(~mask), dev=ds.filter(mask))


def split_heur_length(ds: Dataset, top_pct: float = 0.20) -> DatasetDict:
    """20 % das sentenças mais longas viram conjunto de validação (dev)."""
    lengths = np.array([len(t) for t in ds["tokens"]])
    thr = np.percentile(lengths, 100 * (1 - top_pct))  
    mask = lengths >= thr  

    dev_idx = np.where(mask)[0].tolist()  # índices → list[int]
    train_idx = np.where(~mask)[0].tolist()

    return DatasetDict(
        train=ds.select(train_idx),
        dev=ds.select(dev_idx),
    )

    # Tamanho da sentenças


# def split_heur_rare(ds: Dataset, freq_thr: int = 5) -> DatasetDict:
#     freq = Counter(w.lower() for sent in ds["tokens"] for w in sent)
#     rare = {w for w, c in freq.items() if c <= freq_thr}

#     def has_rare(example):
#         return any(w.lower() in rare for w in example["tokens"])

#     return DatasetDict(
#         train=ds.filter(lambda ex: not has_rare(ex)), dev=ds.filter(has_rare)
#     )


def split_heur_rare(ds: Dataset, freq_thr: int = 5) -> DatasetDict:
    freq = Counter(w.lower() for sent in ds["tokens"] for w in sent)
    rare = {w for w, c in freq.items() if c <= freq_thr}

    keep_dev = []
    for sent in ds["tokens"]:
        print(sent)
        keep_dev.append(any(w.lower() in rare for w in sent))

    dev_idx = [i for i, x in enumerate(keep_dev) if x]
    train_idx = [i for i, x in enumerate(keep_dev) if not x]

    return DatasetDict(
        train=ds.select(train_idx),
        dev=ds.select(dev_idx),
    )

    # Raridade dos tokens

In [10]:
def split_adversarial(ds: Dataset, pct_test: float = 0.20) -> DatasetDict:
    k = int(len(ds) * pct_test)
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    embeds = model.encode([" ".join(t) for t in ds["tokens"]], show_progress_bar=False)
    tree = BallTree(embeds, leaf_size=40)

    idx_train, idx_test = set(range(len(ds))), []
    # semente = ponto mais central
    seed_idx = np.argmax(np.linalg.norm(embeds - embeds.mean(0), axis=1))
    idx_train.remove(seed_idx)
    idx_test.append(seed_idx)
    print(k)
    while len(idx_test) < k:
        print(len(idx_test))
        dists, _ = tree.query(embeds[list(idx_train)], k=1, return_distance=True)
        nxt = list(idx_train)[int(np.argmax(dists))]
        idx_train.remove(nxt)
        idx_test.append(nxt)

    return DatasetDict(
        train=ds.select(sorted(idx_train)),
        dev=ds.select(sorted(idx_test)),
    )

    # Maximizando Wassertein Distance


def split_adversarial_fast(ds: Dataset, pct_test: float = 0.20) -> DatasetDict:
    """
    Farthest-Point Sampling aproximando Wasserstein – versão vetorizada.
    Seleciona pct_test (~20 %) das sentenças como conjunto 'dev'.
    """
    k = int(len(ds) * pct_test)
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

    embeds = model.encode(
        [" ".join(t) for t in ds["tokens"]],
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,  # acelera distância euclidiana ≈ cos
    )

    n = embeds.shape[0]
    idx_all = np.arange(n)

    # 1) ponto mais "central" (norma mais distante da média)
    seed_idx = np.argmax(np.linalg.norm(embeds - embeds.mean(0), axis=1))
    selected = [seed_idx]

    # 2) vetor de distâncias mínimas a qualquer ponto já escolhido
    min_dists = np.linalg.norm(embeds - embeds[seed_idx], axis=1)

    while len(selected) < k:
        next_idx = np.argmax(min_dists)
        selected.append(next_idx)

        # atualiza min_dists com a distância ao novo ponto — tudo de uma vez
        d_new = np.linalg.norm(embeds - embeds[next_idx], axis=1)
        min_dists = np.minimum(min_dists, d_new)

    train_idx = np.setdiff1d(idx_all, selected, assume_unique=True)

    return DatasetDict(
        train=ds.select(train_idx.tolist()),
        dev=ds.select(selected),
    )

In [11]:
# def loc_split(
#     dataset: Dataset, pct_test: float = 0.20, ngram: int = 4, seed: int = 42
# ) -> DatasetDict:
#     """
#     Split baseado em baixa sobreposição léxica (4-gram Jaccard).
#     Teste = pct_test das sentenças com menor overlap em relação ao pool.
#     """
#     # 1. Texto plano por sentença
#     docs = [" ".join(toks) for toks in dataset["tokens"]]

#     # 2. Vetorizar 4-grams (binário)
#     vect = CountVectorizer(
#         analyzer="word", ngram_range=(ngram, ngram), binary=True
#     ).fit(docs)
#     X = vect.transform(docs)

#     # 3. Similaridade Jaccard aproximada com matriz binária
#     # Jaccard(A,B) = |A∩B|/|A∪B| = 1 - |AΔB|/|A∪B|
#     # Usamos: overlap = (A·Bᵀ) / (|A|+|B|-A·Bᵀ)
#     bin_counts = X.sum(axis=1).A1

#     # Para cada doc i, escolhemos vizinho + próximo (fast):
#     from sklearn.metrics.pairwise import cosine_similarity

#     # (cosine no binário ∝ |A∩B|)
#     sim = cosine_similarity(X, dense_output=False)
#     # Soma dos top-k overlaps (k=5) como score
#     k = 5
#     topk = np.zeros(len(dataset))
#     for i in range(sim.shape[0]):
#         row = sim.getrow(i).toarray()[0]
#         idx = np.argpartition(-row, range(1, k + 1))[1 : k + 1]
#         # overlap ≈ |∩|
#         inter = row[idx] * bin_counts[i]
#         uni = bin_counts[i] + bin_counts[idx] - inter
#         topk[i] = (inter / uni).mean()

#     # 4. Ordenar por overlap crescente ⇒ mais “novos” vão p/ teste
#     order = np.argsort(topk)
#     n_test = int(len(dataset) * pct_test)
#     test_idx = order[:n_test]
#     train_idx = order[n_test:]

#     return DatasetDict(
#         {"train": dataset.select(train_idx), "test": dataset.select(test_idx)}
#     )

In [12]:
def loc_split(
    dataset: Dataset,
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    ngram: int = 4,
    seed: int = 42,
) -> DatasetDict:
    """
    Divide por sobreposição léxica (n-gram Jaccard).
    Frações independentes para teste e validação.
    """
    docs = [" ".join(toks) for toks in dataset["tokens"]]

    vect = CountVectorizer(
        analyzer="word", ngram_range=(ngram, ngram), binary=True
    ).fit(docs)
    X = vect.transform(docs)
    bin_counts = X.sum(axis=1).A1

    sim = cosine_similarity(X, dense_output=False)
    k = 5
    topk = np.zeros(len(dataset))
    for i in range(sim.shape[0]):
        row = sim.getrow(i).toarray()[0]
        idx = np.argpartition(-row, range(1, k + 1))[1 : k + 1]
        inter = row[idx] * bin_counts[i]
        uni = bin_counts[i] + bin_counts[idx] - inter
        topk[i] = (inter / uni).mean()

    order = np.argsort(topk)  # baixo → alto overlap
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    test_idx = order[:n_test]
    val_idx = order[n_test : n_test + n_val]
    train_idx = order[n_test + n_val :]

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [13]:
def semantic_cluster_split(
    dataset: Dataset,
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    k: int | None = None,
    seed: int = 42,
) -> DatasetDict:
    """
    Clusters SBERT → reserva clusters distantes para test/val.
    """
    if k is None:
        k = int(np.sqrt(len(dataset)))

    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    embeddings = sbert.encode(
        [" ".join(t) for t in tqdm(dataset["tokens"])],
        batch_size=64,
        show_progress_bar=False,
    )

    km = KMeans(n_clusters=k, random_state=seed, n_init=10).fit(embeddings)
    labels = km.labels_
    centroids = km.cluster_centers_

    global_center = embeddings.mean(0, keepdims=True)
    dists = pairwise_distances(centroids, global_center).flatten()

    clusters_sorted = np.argsort(-dists)  # mais distantes primeiro
    test_clusters, val_clusters = set(), set()
    total_test = total_val = 0
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    for c in clusters_sorted:
        size = np.sum(labels == c)
        if total_test < n_test:  # preenche primeiro o teste
            test_clusters.add(c)
            total_test += size
        elif total_val < n_val:  # depois a validação
            val_clusters.add(c)
            total_val += size
        if total_test >= n_test and total_val >= n_val:
            break

    test_idx = np.where([lbl in test_clusters for lbl in labels])[0]
    val_idx = np.where([lbl in val_clusters for lbl in labels])[0]
    train_idx = np.where(
        [lbl not in test_clusters and lbl not in val_clusters for lbl in labels]
    )[0]

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [14]:
def difficulty_scores(dataset: Dataset) -> np.ndarray:
    length = np.array([len(tok) for tok in dataset["tokens"]], dtype=float)
    length = (length - length.mean()) / length.std()

    dens = []
    for labels in dataset["ner_tags"]:
        non_o = sum(1 for l in labels if l != "O")
        dens.append(non_o / len(labels))
    dens = np.array(dens)
    dens = (dens - dens.mean()) / dens.std()

    ent_types, freq = [], Counter()
    for labels in dataset["ner_tags"]:
        types = [l[2:] for l in labels if l != "O"]
        ent_types.append(types[0] if types else "NONE")
    freq.update(ent_types)
    rarity = np.array([1 / freq[t] for t in ent_types])
    rarity = (rarity - rarity.mean()) / rarity.std()

    return length + dens + rarity


def reverse_curriculum_split(
    dataset: Dataset, pct_test: float = 0.20, pct_val: float = 0.10, seed: int = 42
) -> DatasetDict:
    scores = difficulty_scores(dataset)
    order = np.argsort(scores)  # easy→hard
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    test_idx = order[-n_test:]  # hardest
    val_idx = order[-(n_test + n_val) : -n_test]
    train_idx = order[: -(n_test + n_val)]

    rng = np.random.RandomState(seed)
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [15]:
def heur_len_split(dataset: Dataset,
                   pct_test: float = 0.20,
                   pct_val : float = 0.10,
                   seed: int = 42) -> DatasetDict:
    """Testa só sentenças ≥ máx(len_train)."""
    sent_lens = np.array([len(t) for t in dataset["tokens"]])
    # separação inicial train/dev (randômica estratificando por tamanho grosso)
    idx_all   = np.arange(len(dataset))
    train_idx, temp_idx = train_test_split(idx_all,
                                           test_size=pct_test + pct_val,
                                           stratify=(sent_lens//5),  # bin len
                                           random_state=seed)
    # define longo-threshold como tamanho máx. do treino
    max_train_len = sent_lens[train_idx].max()
    # test = sentenças > threshold.  Caso falte/ sobre exemplos, ajusta.
    test_idx  = [i for i in temp_idx if sent_lens[i] > max_train_len]
    resto_idx = [i for i in temp_idx if i not in test_idx]
    # completa ou reduz para atingir pct_test
    need = int(pct_test*len(dataset)) - len(test_idx)
    if need > 0:
        test_idx.extend(resto_idx[:need])
        val_idx = resto_idx[need:]
    else:
        val_keep = int(pct_val*len(dataset))
        val_idx  = resto_idx[:val_keep]
        test_idx = test_idx[: int(pct_test*len(dataset))]
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 2) Heuristic Rare-Words ----------------------------------
def heur_rare_split(dataset: Dataset,
                    pct_test: float = 0.20,
                    pct_val : float = 0.10,
                    seed: int = 42) -> DatasetDict:
    """Testa frases que contenham palavras do quintil + raro."""
    # contagem de frequência de token
    freqs = Counter(w for toks in dataset["tokens"] for w in toks)
    # define rareza: 20 % mais raras
    thresh = np.quantile(list(freqs.values()), 0.20)
    rare_set = {w for w,c in freqs.items() if c <= thresh}
    is_rare = np.array([any(w in rare_set for w in toks)
                        for toks in dataset["tokens"]])
    rare_idx   = np.where(is_rare)[0]
    common_idx = np.where(~is_rare)[0]
    # garante proporções desejadas
    n_test = int(pct_test*len(dataset))
    n_val  = int(pct_val *len(dataset))
    rng = np.random.default_rng(seed)
    test_idx = rng.choice(rare_idx, size=min(len(rare_idx), n_test),
                          replace=False)
    resto_idx = [i for i in rare_idx if i not in test_idx] + list(common_idx)
    val_idx  = rng.choice(resto_idx, size=n_val, replace=False)
    train_idx = [i for i in resto_idx if i not in val_idx]
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 3) Standard (80-10-10) -----------------------------------
def std_split(dataset: Dataset,
              pct_test: float = 0.10,
              pct_val : float = 0.10,
              seed: int = 42) -> DatasetDict:
    """Split aleatório estratificado por comprimento (PTB-like)."""
    idx = np.arange(len(dataset))
    strat = (np.array([len(t) for t in dataset["tokens"]]) // 5)
    train_idx, temp_idx = train_test_split(idx, test_size=pct_test+pct_val,
                                          stratify=strat, random_state=seed)
    val_rel = pct_val / (pct_test+pct_val)
    val_idx, test_idx = train_test_split(temp_idx, test_size=1-val_rel,
                                         stratify=strat[temp_idx],
                                         random_state=seed)
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 4) Adversarial (approx. Wasserstein) ---------------------
def adversarial_split(dataset: Dataset,
                      pct_test: float = 0.20,
                      pct_val : float = 0.10,
                      k: int = 5,
                      seed: int = 42) -> DatasetDict:
    """
    Seleciona k frases 'mais distantes' recursivamente (BallTree + W₂)
    para compor o teste, lembrando Alg.-1 de Søgaard et al.【turn6file4】.
    """
    # SBERT embed (rápido na GPU / aceitável CPU)
    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    emb = sbert.encode([" ".join(toks) for toks in dataset["tokens"]],
                       batch_size=64, show_progress_bar=False)
    # BallTree para vizinhança eficiente
    tree = BallTree(emb, leaf_size=40)
    idx_pool = set(range(len(dataset)))
    test_idx = []
    rng = np.random.default_rng(seed)
    while len(test_idx) < int(pct_test*len(dataset)):
        # amostra candidata: ponto + longe do centro
        center = emb[list(idx_pool)].mean(0, keepdims=True)
        dists, _ = tree.query(center, k=len(idx_pool))
        farthest = list(idx_pool)[int(dists.argmax())]
        # pega-se farest e seus k-NN mais próximos  ⇒ aumenta diversidade
        nn = tree.query([emb[farthest]], k=k, return_distance=False)[0]
        for j in nn:
            if j in idx_pool and len(test_idx) < int(pct_test*len(dataset)):
                test_idx.append(j)
                idx_pool.remove(j)
    # retira val
    val_size = int(pct_val*len(dataset))
    val_idx  = rng.choice(list(idx_pool), size=val_size, replace=False)
    idx_pool -= set(val_idx)
    train_idx = list(idx_pool)
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [16]:
def heur_len_split(dataset: Dataset,
                   pct_test: float = 0.20,
                   pct_val : float = 0.10,
                   seed: int = 42,
                   bin_size: int = 10) -> DatasetDict:
    """
    Teste = sentenças mais longas que max(len(train)).
    Robusto a datasets pequenos: se estratificação falhar, usa split aleatório.
    """
    rng = np.random.default_rng(seed)
    idx_all   = np.arange(len(dataset))
    sent_lens = np.array([len(t) for t in dataset["tokens"]])

    # ------ 1) tenta split estratificado por baldes -------------------------
    strat = (sent_lens // bin_size)
    try:
        train_idx, temp_idx = train_test_split(
            idx_all,
            test_size=pct_test + pct_val,
            stratify=strat,
            random_state=seed,
        )
    except ValueError:                       # classes com 1 amostra
        train_idx, temp_idx = train_test_split(
            idx_all,
            test_size=pct_test + pct_val,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    # ------ 2) escolhe test = len > max(train) ------------------------------
    max_train_len = sent_lens[train_idx].max()
    test_idx  = [i for i in temp_idx if sent_lens[i] > max_train_len]
    resto_idx = [i for i in temp_idx if i not in test_idx]

    # garante tamanhos exatos
    n_test_desired = int(pct_test * len(dataset))
    n_val_desired  = int(pct_val  * len(dataset))

    # completa teste se ficou pequeno
    if len(test_idx) < n_test_desired:
        extra = rng.choice(resto_idx,
                           size=n_test_desired - len(test_idx),
                           replace=False)
        test_idx.extend(extra)
        resto_idx = [i for i in resto_idx if i not in extra]

    # define validação
    val_idx  = rng.choice(resto_idx, size=n_val_desired, replace=False)
    train_idx = [i for i in idx_all if i not in test_idx and i not in val_idx]

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [17]:
def std_split(dataset,
              pct_test: float = 0.10,
              pct_val : float = 0.10,
              seed: int = 42,
              bin_size: int = 5) -> DatasetDict:
    """
    Split 80-10-10 robusto.
      • Tenta estratificar por comprimento // bin_size.
      • Se houver classes com <2 amostras, recua p/ split aleatório.
    """
    idx  = np.arange(len(dataset))
    bins = (np.array([len(t) for t in dataset["tokens"]]) // bin_size)

    try:
        train_idx, temp_idx = train_test_split(
            idx,
            test_size=pct_test + pct_val,
            stratify=bins,
            random_state=seed,
        )
    except ValueError:                       # classes muito pequenas
        train_idx, temp_idx = train_test_split(
            idx,
            test_size=pct_test + pct_val,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    # fraciona temp em val / test mantendo proporção desejada
    val_share = pct_val / (pct_test + pct_val)
    try:
        val_idx, test_idx = train_test_split(
            temp_idx,
            test_size=1 - val_share,
            stratify=bins[temp_idx],
            random_state=seed,
        )
    except ValueError:
        val_idx, test_idx = train_test_split(
            temp_idx,
            test_size=1 - val_share,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [18]:
def adversarial_split(dataset: Dataset,
                      pct_test: float = 0.20,
                      pct_val : float = 0.10,
                      k: int = 5,
                      seed: int = 42) -> DatasetDict:
    """
    Versão robusta: nunca “trava” antes de atingir n_test.
    Seleciona blocos de k sentenças mais distantes do centro iterativamente.
    """
    # -------------------------------- embeds -------------------------------
    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    emb = sbert.encode([" ".join(t) for t in dataset["tokens"]],
                       batch_size=64, show_progress_bar=False)
    tree = BallTree(emb, leaf_size=40)

    n_test = int(pct_test * len(dataset))
    n_val  = int(pct_val  * len(dataset))

    idx_pool = set(range(len(dataset)))
    test_idx = []
    rng = np.random.default_rng(seed)

    print(f"Selecionando {n_test} sentenças para teste…")
    step = 0
    while len(test_idx) < n_test and idx_pool:
        if step % 5 == 0:
            print(f"  {len(test_idx)} selecionadas…")
        step += 1

        pool_list = list(idx_pool)
        center = emb[pool_list].mean(0, keepdims=True)

        # distância euclidiana ao centro global restante
        dists, _ = tree.query(center, k=len(pool_list))
        farthest_local_idx = int(dists.argmax())
        farthest_global_idx = pool_list[farthest_local_idx]

        # número efetivo de vizinhos
        k_eff = min(k, len(idx_pool))
        nn = set(tree.query([emb[farthest_global_idx]],
                            k=k_eff,
                            return_distance=False)[0])

        # adiciona vizinhos ainda não selecionados
        for j in nn:
            if j in idx_pool and len(test_idx) < n_test:
                test_idx.append(j)
                idx_pool.remove(j)

        # Se nada foi adicionado (pode acontecer quando sobram <k únicos)
        if farthest_global_idx not in test_idx:
            test_idx.append(farthest_global_idx)
            idx_pool.remove(farthest_global_idx)

    # ------------------------ validação e treino ---------------------------
    val_idx = rng.choice(list(idx_pool), size=n_val, replace=False)
    idx_pool -= set(val_idx)
    train_idx = list(idx_pool)

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [19]:
standard_split = std_split(geocorpus_full)
print('std')
# random_splt = random_splits(geocorpus_full)
# print('random')
heur_len = heur_len_split(geocorpus_full)
print("heur_len")
heur_rare = heur_rare_split(geocorpus_full)
print("heur_rare")
advers = adversarial_split(geocorpus_full)
print("advs")
loc = loc_split(geocorpus_full)
print("loc")
semantic = semantic_cluster_split(geocorpus_full)
print("semantic")
reverse = reverse_curriculum_split(geocorpus_full)
print("reverse")

std
heur_len
heur_rare
Selecionando 1054 sentenças para teste…
  0 selecionadas…
  24 selecionadas…
  47 selecionadas…
  71 selecionadas…
  96 selecionadas…
  121 selecionadas…
  144 selecionadas…
  165 selecionadas…
  187 selecionadas…
  211 selecionadas…
  232 selecionadas…
  251 selecionadas…
  271 selecionadas…
  288 selecionadas…
  305 selecionadas…
  329 selecionadas…
  349 selecionadas…
  364 selecionadas…
  383 selecionadas…
  399 selecionadas…
  417 selecionadas…
  433 selecionadas…
  452 selecionadas…
  470 selecionadas…
  488 selecionadas…
  501 selecionadas…
  515 selecionadas…
  529 selecionadas…
  542 selecionadas…
  554 selecionadas…
  565 selecionadas…
  577 selecionadas…
  590 selecionadas…
  597 selecionadas…
  610 selecionadas…
  628 selecionadas…
  642 selecionadas…
  658 selecionadas…
  671 selecionadas…
  685 selecionadas…
  698 selecionadas…
  713 selecionadas…
  724 selecionadas…
  738 selecionadas…
  754 selecionadas…
  771 selecionadas…
  785 selecionadas…
  7

100%|██████████| 5272/5272 [00:00<00:00, 1199672.89it/s]


semantic
reverse


# Pipeline Ollama

In [20]:
def tokens_to_text(tokens: List[str]) -> str:
    # Junta preservando espaços simples.
    return " ".join(tokens)


def fmt_example(ex: dict) -> str:
    toks = ex["tokens"]
    tags = ex.get("tags") or ex.get("ner_tags")  # compatível com seus dsets
    pairs_line = " ".join(f"{i+1}-{t}" for i, t in enumerate(tags))
    return f"Tokens: {' '.join(toks)}\nSaída: {pairs_line}"

def build_prompt(few_shot: List[Dict[str, Any]], query_tokens: List[str]) -> str:
    N = len(query_tokens)
    header = f"""
        Você é um anotador especialista em NER. Rotule **cada token** usando o esquema **IOB2**.

        Regras (case-sensitive):
        1) Use apenas os rótulos:\n{iob_labels}

        2) IOB2 (cheque silenciosamente):
        • Uma entidade inicia em B-<TIPO>.
        • I-<TIPO> só após B-<TIPO> ou I-<TIPO> do mesmo TIPO.
        • Nunca iniciar com I-.

        3) Formato ÚNICO de saída (sem texto extra):
        • Uma única linha com N pares índice-rótulo (1-based), separados por espaço.
        • Cada par: <índice>-<RÓTULO>.
        • Ex.: 1-O 2-B-magmaticas 3-I-magmaticas ... N-O

        4) Se houver N tokens, produza exatamente N rótulos.
        """.strip()

    examples = "\n\n### Exemplos\n" + "\n\n".join(fmt_example(ex) for ex in few_shot)

    task = f"""
        ### Tarefa
        Tokens: {tokens_to_text(query_tokens)}

        ### Saída esperada
        Retorne **apenas**: {{"tags": ["O", "B-...", "..."]}} com {N} itens.
        """.strip()

    return f"{header}\n\n{examples}\n\n{task}"


In [21]:
def select_few_shot(exemplars: Dataset, k=FEW_SHOT_K) -> list[dict]:
    # amostra k exemplos aleatórios (pode trocar por stratified sampling)
    return random.sample(list(exemplars), k)

In [22]:
def ollama_generate(
    model: str,
    prompt: str,
    options: Optional[Dict[str, Any]] = None,
    host: str = "http://localhost:11434",
    timeout: int = 120,
    stream: bool = False,
) -> str:
    """
    Wrapper do endpoint /api/generate.
    """
    url = f"{host}/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": stream,
        "options": options or {},
        # "stop": ["\n\n"],  # ajuste opcional se algum modelo insistir em linhas extras
    }
    # Timeout levemente alto pois alguns modelos podem demorar.
    resp = requests.post(url, json=payload, timeout=timeout)
    resp.raise_for_status()

    if stream:
        # Se usar stream=True, agregamos as linhas 'data: {...}'
        text = ""
        for line in resp.iter_lines():
            if not line:
                continue
            try:
                obj = json.loads(line.decode("utf-8"))
            except Exception:
                continue
            text += obj.get("response", "")
        return text
    else:
        data = resp.json()
        return data.get("response", "")

In [23]:
def predict_labels_ollama(few_shot, query_tokens, model_key=MODEL_KEY, host=HOST):
    cfg = OLLAMA_MODELS[model_key]
    prompt = build_prompt(few_shot, query_tokens)
    print(prompt)
    raw = ollama_generate(model=model_key, prompt=prompt, options=cfg.get("options"), host=host, stream=False)
    return raw

In [24]:
SESSION = requests.Session()

In [25]:
def ollama_generate_json(model: str, prompt: str, host="http://127.0.0.1:11434",
                         num_predict:int=64, num_ctx:int=2048, timeout=(10, 600)):
    payload = {
        "model": model,
        "prompt": prompt,
        "format": "json",     # resposta vem como string JSON em data['response']
        "stream": False,      # evita travar se o cliente não consumir stream
        "keep_alive": "30m",
        "options": {
            "temperature": 0,
            "seed": 42,
            "num_ctx": num_ctx*30,
            "num_predict": max(64, 5 * num_predict)
        }
    }
    return SESSION.post(f"{host}/api/generate", json=payload, timeout=timeout)

In [26]:
import time

In [27]:
def predict_labels_ollama(tokens: List[str],
                          few_shot_text: str = "",
                          model: str = "llama3.1:8b") -> List[str]:
    # prompt curto e objetivo; evite few-shots gigantes (cole 1–2 exemplos no few_shot_text)
    
    N = len(tokens)
    prompt = build_prompt(few_shot_text, tokens)
    # retry leve para 5xx/timeout
    for attempt in range(3):
        try:
            r = ollama_generate_json(
                model=model,
                prompt=prompt,
                # use o número de tokens como base para calcular num_predict interno
                num_predict=min(64, 3 * len(tokens)),
                num_ctx=N
            )
            r.raise_for_status()
            data = r.json()
            attempt = 3
            return data['response']
        except Exception:
            if attempt == 2:
                raise
            time.sleep(1.5 * (attempt + 1))

# Prompts

In [28]:
# few_shot = [
#     {"tokens": ["o", "maj", "é", "composto"], "tags": ["O","O","O","O"]},
#     {"tokens": ["dioritos", "e", "monzodioritos"], "tags": ["B-magmaticas","O","I-magmaticas"]},
# ]
# query_tokens = ["o","maj","é","composto","por","monzodioritos","."]
# preds = predict_labels_ollama(few_shot, query_tokens)  # usa MODEL_KEY
# preds

In [29]:
standard_split = std_split(geocorpus_full)
print('std')
# random_splt = random_splits(geocorpus_full)
# print('random')
heur_len = heur_len_split(geocorpus_full)
print("heur_len")
heur_rare = heur_rare_split(geocorpus_full)
print("heur_rare")
advers = adversarial_split(geocorpus_full)
print("advs")
loc = loc_split(geocorpus_full)
print("loc")
semantic = semantic_cluster_split(geocorpus_full)
print("semantic")
reverse = reverse_curriculum_split(geocorpus_full)
print("reverse")

std
heur_len
heur_rare
Selecionando 1054 sentenças para teste…
  0 selecionadas…
  24 selecionadas…
  47 selecionadas…
  71 selecionadas…
  96 selecionadas…
  121 selecionadas…
  144 selecionadas…
  165 selecionadas…
  187 selecionadas…
  211 selecionadas…
  232 selecionadas…
  251 selecionadas…
  271 selecionadas…
  288 selecionadas…
  305 selecionadas…
  329 selecionadas…
  349 selecionadas…
  364 selecionadas…
  383 selecionadas…
  399 selecionadas…
  417 selecionadas…
  433 selecionadas…
  452 selecionadas…
  470 selecionadas…
  488 selecionadas…
  501 selecionadas…
  515 selecionadas…
  529 selecionadas…
  542 selecionadas…
  554 selecionadas…
  565 selecionadas…
  577 selecionadas…
  590 selecionadas…
  597 selecionadas…
  610 selecionadas…
  628 selecionadas…
  642 selecionadas…
  658 selecionadas…
  671 selecionadas…
  685 selecionadas…
  698 selecionadas…
  713 selecionadas…
  724 selecionadas…
  738 selecionadas…
  754 selecionadas…
  771 selecionadas…
  785 selecionadas…
  7

100%|██████████| 5272/5272 [00:00<00:00, 1252754.56it/s]


semantic
reverse


In [284]:
splits = [standard_split, heur_len, heur_rare, advers]

In [285]:
splits_names = ['standard', 'heur_len', 'heur_rare', 'adversarial']

In [309]:
model_escolhido = 'llama3.1:8b'  # ou 'vicuna-13b-v1.5:8b', etc.

In [ ]:
golds, predictions = {}, {}
for split, nome in zip(splits, splits_names):
    split_base = split
    
    #max_len_split = len(split_base["test"])
    max_len_split = 500
    random.seed(SEED_GLOBAL)
    realizado = 0
    few_shot = select_few_shot(split_base["train"], FEW_SHOT_K)
    total = 500 #len(split_base['test'])
    golds[nome] = []
    predictions[nome] = []
    for ex in split_base['test'].select(range(500)):

        print('---', nome, '---')
        pred_tags = predict_labels_ollama(ex["tokens"], few_shot_text=few_shot, model=model_escolhido)
        golds[nome].append([ex["ner_tags"]])
        predictions[nome].append([pred_tags])


        realizado += 1
        clear_output(wait=True)
        print(f'Total de exemplos processados: {realizado/total*100:.2f}%')

Total de exemplos processados: 100.00%


In [316]:
report = {}
f1_macro = {}
precision = {}
recall = {}
metrics_all = {}

bads_sets = {}

for name, predidction in predictions.items():
    preds = []
    print(predidction)
    for tags_prediction in predidction:
        try:
            preds.append(json.loads(tags_prediction)['tags']) 
        except:
            try:
                preds.append(json.loads(tags_prediction[0])['tags']) 
            except:
                try:
                    preds.append(json.loads(tags_prediction[0][0])['tags']) 
                except:
                    preds.append(['O'])
          
    new_preds = []
    for pred_token in preds:
        if type(pred_token) == str:
            new_preds.append(pred_token)
        else:
            tokens_ajustados = []  
            for token in pred_token:
                try:
                    if int(_).is_integer():
                        tokens_ajustados.extend([token_ajustado])
                    else:
                        tokens_ajustados.extend([token])
                except:
                    try:
                        token_ajustado = token['tag']
                        tokens_ajustados.extend([token_ajustado])
                    except:
                        token_ajustado = token
                        tokens_ajustados.extend([token_ajustado])
            new_preds.append(tokens_ajustados)

        
    if type(golds[name][0][0]) == list:
        actual_golds = [gold[0] for gold in golds[name]]
        print('ok')
    else:
        actual_golds = golds[name]
    actual_preds = new_preds
    print(new_preds)
    # clean_golds, cleaned, clean_preds = [], [], []

    # for i, (g, p) in enumerate(zip(actual_golds, actual_preds)):
    #     if type(g[0][0]) == list:
    #         g_ = g[0]
    #         print(len(g_), len(p))
    #         if len(p) < len(g_):
    #             print('limpou preds'); print(i)
    #             cleaned.append(i)
    #             continue  # pula a adição
    #         elif len(p) > len(g_):
    #             print('limpou golds'); print(i)
    #             cleaned.append(i)
    #             continue  # pula a adição
    #     else:
    #         print(len(g), len(p))
    #         if len(p) < len(g):
    #             print('limpou preds'); print(i)
    #             cleaned.append(i)
    #             continue  # pula a adição
    #         elif len(p) > len(g):
    #             print('limpou golds'); print(i)
    #             cleaned.append(i)
    #             continue  # pula a adição

    clean_golds, clean_preds = [], []
    for g, p in zip(actual_golds, actual_preds):
        # se o modelo gerar menos/more tags que tokens, ajustamos:
        if type(g[0][0]) == list:
            g_ = g[0]
            if len(p) < len(g_):
                print('limpou preds')
                p = p + ["O"] * (len(g_) - len(p))          # completa com O
            elif len(p) > len(g_):
                print('limpou golds')
                p = p[:len(g_)]                             # descarta excedente
            clean_golds.append(g_)
            clean_preds.append(p)
        else:
            if len(p) < len(g):
                print('limpou preds')
                p = p + ["O"] * (len(g) - len(p))          # completa com O
            elif len(p) > len(g):
                print('limpou golds')
                p = p[:len(g)]                             # descarta excedente
            clean_golds.append(g)
            clean_preds.append(p)

    # bad = set(cleaned)

    # bads_sets[name] = bad

    # clean_golds[:] = [g for i, g in enumerate(actual_golds) if i not in bad]
    # clean_preds[:] = [p for i, p in enumerate(actual_preds) if i not in bad]

    print(f"{name}: {len(clean_golds)} exemplos válidos ({len(bad)} removidos)")
    print(clean_golds, len(clean_golds))
    print(clean_preds, len(clean_preds))
    if len(clean_golds) != 0 and len(clean_preds) != 0:
        report[name] = classification_report(clean_golds, clean_preds, output_dict=True)
        f1_macro[name] = f1_score(clean_golds, clean_preds, average="macro")
        precision[name] = precision_score(clean_golds, clean_preds, average="macro")
        recall[name] = recall_score(clean_golds, clean_preds, average="macro")
        rep = classification_report(clean_golds, clean_preds, scheme=IOB2, zero_division=0, output_dict=True)

        metrics = {
            "precision_micro":   rep["micro avg"]["precision"],
            "recall_micro":      rep["micro avg"]["recall"],
            "f1_micro":          rep["micro avg"]["f1-score"],
            "precision_macro":   rep["macro avg"]["precision"],
            "recall_macro":      rep["macro avg"]["recall"],
            "f1_macro":          rep["macro avg"]["f1-score"],
            "precision_weighted":rep["weighted avg"]["precision"],
            "recall_weighted":   rep["weighted avg"]["recall"],
            "f1_weighted":       rep["weighted avg"]["f1-score"],
            "accuracy":          accuracy_score(clean_golds, clean_preds),
        }

        metrics_all[name] = metrics

[['{"tags": ["O", "B-ambienteSedimentacao", "sedimentos", "quartzosos", "em", "bacias", "tafrogênicas", "depositados", "em", "ambientes", "continentais", "ambientes", "fluvial", "e", "eólico", "passaram", "a", "ter", "grande", "importância", "na", "história", "da", "terra", "durante", "o", "paleomesoproterozoico"]}'], ['{"tags": ["O", "B-periodo", "B-sedimentaresSiliciclasticas", "I-estratigrafia", "B-estratigrafia", "B-ambienteSedimentacao", "B-sedimentaresSiliciclasticas", "B-sedimentaresSiliciclasticas", "O", "B-estratigrafia", "I-estratigrafia", "B-estratigrafia", "B-periodo", "B-estratigrafia", "I-estratigrafia", "B-sedimentaresSiliciclasticas", "O", "B-estratigrafia", "I-estratigrafia", "B-estratigrafia", "B-periodo", "B-estratigrafia", "I-estratigrafia", "B-sedimentaresSiliciclasticas", "O", "B-estratigrafia", "I-estratigrafia", "B-estratigrafia", "B-periodo", "B-estratigrafia", "I-estratigrafia", "B-sedimentaresSiliciclasticas", "O", "B-estratigrafia", "I-estratigrafia", "B-est

c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: sedimentos seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: quartzosos seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: bacias seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: tafrogênicas seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\seqeval\metrics\sequence_labeling.py:

[['{"tags": ["O", "B-idade", "periodo", "era", "eon", "O", "B-periodo", "O", "B-era", "O", "B-eon", "O", "B-idade", "O", "B-periodo", "O", "B-era", "O", "B-eon", "O", "B-epoca", "O"]}'], ['{"tags": ["O", "B-idade", "periodo", "era", "eon", "O", "B-epoca", "O", "B-idade", "O", "B-periodo", "O", "B-era", "O", "B-eon", "O", "B-idade", "O", "B-periodo", "O", "B-era", "O", "B-eon", "O", "B-epoca", "O", "B-idade", "O", "B-periodo", "O", "B-era", "O", "B-eon"]}'], ['{"tags": ["1-O", "2-O", "3-O", "4-O", "5-O", "6-O", "7-O", "8-O", "9-O", "10-O", "11-O", "12-O", "13-B-epoca", "14-O"]}'], ['{"tags": ["O", "B-idade", "epoca", "periodo", "era", "eon", "idade", "periodo", "era", "eon", "idade", "periodo", "era", "eon", "idade", "periodo", "era", "eon", "O", "B-epoca", "O"]}'], ['{"tags": ["O", "B-idade", "O", "B-epoca", "O", "B-eon", "O", "B-periodo", "O", "B-era", "O", "B-epoca", "O", "B-idade", "O", "B-periodo", "O", "B-era", "O", "B-eon", "O", "B-idade", "O", "B-epoca", "O", "B-epoca"]}'], ['{"

c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: periodo seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: era seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: eon seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: 13-B-epoca seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarnin

[['{"tags": ["O", "B-idade", "O", "B-episódio", "O", "B-conformidade", "O", "B-orogenia", "O", "B-clivagem", "O", "B-dobras", "O", "B-deformação", "O", "B-associadas", "O", "B-corresponde", "O", "B-existente", "O", "B-discordância", "O", "B-datada", "O", "B-givetiano", "O", "B-pedreira", "O", "B-engenharia", "O", "B-cabrela"]}'], ['{"tags": ["O", "B-unidadeEstratigrafica", "B-contextoGeologicoDeBacia", "B-epoca", "B-idade", "B-..."]}'], ['{"tags": ["O", "B-unidadeEstratigrafica", "B-contextoGeologicoDeBacia", "B-epoca", "B-idade", "B-unidadeEstratigrafica", "B-contextoGeologicoDeBacia", "B-epoca"]}'], ['{"tags": ["O", "B-idade", "B-episódio", "B-conformidade", "B-região", "B-sistema", "B-sequência", "B-unidades", "B-tipo", "B-rift", "B-sag", "B-pedreira", "B-deformação", "B-orogenia", "B-varisca", "B-clivagem", "B-dobras", "B-associadas", "B-corresponde", "B-existente", "B-discordância", "B-formações", "B-pedreira", "B-engenharia", "B-cabrela", "B-givetiano", "B-datada", "B-episódio", 

c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: Grego seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: phneros seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: visível seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: zoikos seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarn

[['{"tags": ["O", "B-idade", "época", "era", "período", "siluriano", "sucede", "precede", "idade", "hirauntiana", "rhuddaniana", "llandovery", "sucedem-se", "epocas", "eram", "periodos", "silurianos", "era", "período", "siluriano", "sucessão", "precedência", "idade", "hirauntiana", "rhuddaniana", "llandovery", "sucedem-se", "epocas", "eram", "periodos", "silurianos"]}'], ['{"tags": ["O", "B-idade", "epoca", "periodo", "era", "eono", "sede", "sedimento", "sedimentares", "carbonatos", "abertura", "oceano", "atlantico", "terciario", "oligoceno", "presenca", "argilas", "quantidade", "continua", "atlantico", "tipos", "litologicos", "rochas", "vulcanicas", "intrusivas", "vulcano-clasticas", "autoclasicas", "piroclasticas", "ultrabasicas", "idades", "radiometricas", "ar-ar", "paleogeno", "paleoceno", "meso-eoceno", "significa", "perigo", "apedeutismo", "população", "mundo", "civilizada", "educada"]}'], ['{"tags": ["O", "B-epoca", "O", "B-epoca", "O", "B-epoca", "O", "B-epoca", "O", "B-epoca",

c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: época seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: siluriano seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: hirauntiana seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: sucedem-se seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171

In [317]:
report

{'standard': {'...': {'precision': np.float64(0.0),
   'recall': np.float64(0.0),
   'f1-score': np.float64(0.0),
   'support': np.int64(0)},
  '0': {'precision': np.float64(0.0),
   'recall': np.float64(0.0),
   'f1-score': np.float64(0.0),
   'support': np.int64(0)},
  '004': {'precision': np.float64(0.0),
   'recall': np.float64(0.0),
   'f1-score': np.float64(0.0),
   'support': np.int64(0)},
  '010': {'precision': np.float64(0.0),
   'recall': np.float64(0.0),
   'f1-score': np.float64(0.0),
   'support': np.int64(0)},
  '12': {'precision': np.float64(0.0),
   'recall': np.float64(0.0),
   'f1-score': np.float64(0.0),
   'support': np.int64(0)},
  '50': {'precision': np.float64(0.0),
   'recall': np.float64(0.0),
   'f1-score': np.float64(0.0),
   'support': np.int64(0)},
  '996': {'precision': np.float64(0.0),
   'recall': np.float64(0.0),
   'f1-score': np.float64(0.0),
   'support': np.int64(0)},
  'B-': {'precision': np.float64(0.0),
   'recall': np.float64(0.0),
   'f1-score'

In [318]:
f1_macro

{'standard': np.float64(8.886333858486097e-05),
 'heur_len': np.float64(2.8265317305228006e-05),
 'heur_rare': np.float64(6.668953242691234e-05),
 'adversarial': np.float64(5.6403577802502874e-05)}

In [319]:
precision

{'standard': np.float64(7.707353096642911e-05),
 'heur_len': np.float64(1.602717638395342e-05),
 'heur_rare': np.float64(3.704846731806967e-05),
 'adversarial': np.float64(3.873963659561645e-05)}

In [320]:
recall

{'standard': np.float64(0.00021936631390695918),
 'heur_len': np.float64(0.00012397304711497865),
 'heur_rare': np.float64(0.0003468655246598865),
 'adversarial': np.float64(0.000191664391110745)}

In [321]:
precision_micro = []
recall_micro = []
f1_micro = []
f1_macro = []
precision_macro = []
recall_macro = []
f1_weighted = []
precision_weighted = []
recall_weighted = []
accuracy = []
name = []

for keys,value in metrics_all.items():
    name.append(keys)
    precision_micro.append(value['precision_micro'])
    recall_micro.append(value['recall_micro'])
    f1_micro.append(value['f1_micro'])
    precision_macro.append(value['precision_macro'])
    recall_macro.append(value['recall_macro'])
    f1_macro.append(value['f1_macro'])
    precision_weighted.append(value['precision_weighted'])
    recall_weighted.append(value['recall_weighted'])
    f1_weighted.append(value['f1_weighted'])
    accuracy.append(value['accuracy'])

df_results = pd.DataFrame({'name':name,
                           'precision_micro':precision_micro,
                           'recall_micro':recall_micro,
                           'f1_micro':f1_micro,
                           'precision_macro':precision_macro,
                           'recall_macro':recall_macro,
                           'f1_macro':f1_macro,
                           'precision_weighted':precision_weighted,
                           'recall_weighted':recall_weighted,
                           'f1_weighted':f1_weighted,
                           'accuracy':accuracy})


In [322]:
df_results

,name,precision_micro,recall_micro,f1_micro,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,accuracy
0,standard,0.002979,0.011494,0.004731,0.000077,0.000219,0.000089,0.004425,0.011494,0.005216,0.699315
1,heur_len,0.001642,0.012528,0.002904,0.000016,0.000124,0.000028,0.001605,0.012528,0.002833,0.529615
2,heur_rare,0.002808,0.016298,0.004790,0.000037,0.000347,0.000067,0.001753,0.016298,0.003154,0.656876
3,adversarial,0.001474,0.005476,0.002322,0.000039,0.000192,0.000056,0.001068,0.005476,0.001577,0.658037


In [323]:
df_results.to_csv(f'metricas_{model_escolhido}_exclusao.csv', index=False)

In [363]:
report = {}
f1_macro = {}
precision = {}
recall = {}
metrics_all2 = {}

bads_sets = {}

for name, predidction in predictions.items():
    preds = []
    print(predidction)
    for tags_prediction in predidction:
        try:
            preds.append(json.loads(tags_prediction)['tags']) 
        except:
            try:
                preds.append(json.loads(tags_prediction[0])['tags']) 
            except:
                try:
                    preds.append(json.loads(tags_prediction[0][0])['tags']) 
                except:
                    preds.append(['O'])
          
    new_preds = []
    for pred_token in preds:
        if type(pred_token) == str:
            new_preds.append(pred_token)
        else:
            tokens_ajustados = []  
            for token in pred_token:
                try:
                    if int(_).is_integer():
                        tokens_ajustados.extend([token_ajustado])
                    else:
                        tokens_ajustados.extend([token])
                except:
                    try:
                        token_ajustado = token['tag']
                        tokens_ajustados.extend([token_ajustado])
                    except:
                        token_ajustado = token
                        tokens_ajustados.extend([token_ajustado])
            new_preds.append(tokens_ajustados)

        
    if type(golds[name][0][0]) == list:
        actual_golds = [gold[0] for gold in golds[name]]
        print('ok')
    else:
        actual_golds = golds[name]
    actual_preds = new_preds
    print(new_preds)
    clean_golds, cleaned, clean_preds = [], [], []

    for i, (g, p) in enumerate(zip(actual_golds, actual_preds)):
        if type(g[0][0]) == list:
            g_ = g[0]
            print(len(g_), len(p))
            if len(p) < len(g_):
                print('limpou preds'); print(i)
                cleaned.append(i)
                continue  # pula a adição
            elif len(p) > len(g_):
                print('limpou golds'); print(i)
                cleaned.append(i)
                continue  # pula a adição
        else:
            print(len(g), len(p))
            if len(p) < len(g):
                print('limpou preds'); print(i)
                cleaned.append(i)
                continue  # pula a adição
            elif len(p) > len(g):
                print('limpou golds'); print(i)
                cleaned.append(i)
                continue  # pula a adição

    bad = set(cleaned)

    bads_sets[name] = bad

    clean_golds[:] = [g for i, g in enumerate(actual_golds) if i not in bad]
    clean_preds[:] = [p for i, p in enumerate(actual_preds) if i not in bad]


    if len(clean_golds) != 0 and len(clean_preds) != 0:
        report[name] = classification_report(clean_golds, clean_preds, output_dict=True)
        f1_macro[name] = f1_score(clean_golds, clean_preds, average="macro")
        precision[name] = precision_score(clean_golds, clean_preds, average="macro")
        recall[name] = recall_score(clean_golds, clean_preds, average="macro")

        rep = classification_report(clean_golds, clean_preds, scheme=IOB2, zero_division=0, output_dict=True)
        
        metrics2 = {
            "precision_micro":   rep["micro avg"]["precision"],
            "recall_micro":      rep["micro avg"]["recall"],
            "f1_micro":          rep["micro avg"]["f1-score"],
            "precision_macro":   rep["macro avg"]["precision"],
            "recall_macro":      rep["macro avg"]["recall"],
            "f1_macro":          rep["macro avg"]["f1-score"],
            "precision_weighted":rep["weighted avg"]["precision"],
            "recall_weighted":   rep["weighted avg"]["recall"],
            "f1_weighted":       rep["weighted avg"]["f1-score"],
            "accuracy":          accuracy_score(clean_golds, clean_preds),
        }

        metrics_all2[name] = metrics2

[['{"tags": ["O", "B-ambienteSedimentacao", "sedimentos", "quartzosos", "em", "bacias", "tafrogênicas", "depositados", "em", "ambientes", "continentais", "ambientes", "fluvial", "e", "eólico", "passaram", "a", "ter", "grande", "importância", "na", "história", "da", "terra", "durante", "o", "paleomesoproterozoico"]}'], ['{"tags": ["O", "B-periodo", "B-sedimentaresSiliciclasticas", "I-estratigrafia", "B-estratigrafia", "B-ambienteSedimentacao", "B-sedimentaresSiliciclasticas", "B-sedimentaresSiliciclasticas", "O", "B-estratigrafia", "I-estratigrafia", "B-estratigrafia", "B-periodo", "B-estratigrafia", "I-estratigrafia", "B-sedimentaresSiliciclasticas", "O", "B-estratigrafia", "I-estratigrafia", "B-estratigrafia", "B-periodo", "B-estratigrafia", "I-estratigrafia", "B-sedimentaresSiliciclasticas", "O", "B-estratigrafia", "I-estratigrafia", "B-estratigrafia", "B-periodo", "B-estratigrafia", "I-estratigrafia", "B-sedimentaresSiliciclasticas", "O", "B-estratigrafia", "I-estratigrafia", "B-est

c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: 1-O seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: 2-O seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: 3-O seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: 4-O seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: 5-O seem

22 23
limpou golds
479
28 15
limpou preds
480
14 14
25 26
limpou golds
482
20 20
47 9
limpou preds
484
22 15
limpou preds
485
52 20
limpou preds
486
43 1
limpou preds
487
55 1
limpou preds
488
32 30
limpou preds
489
21 21
21 8
limpou preds
491
11 11
45 22
limpou preds
493
13 1
limpou preds
494
63 14
limpou preds
495
28 30
limpou golds
496
43 17
limpou preds
497
25 26
limpou golds
498
34 12
limpou preds
499
[['{"tags": ["O", "B-idade", "O", "B-episódio", "O", "B-conformidade", "O", "B-orogenia", "O", "B-clivagem", "O", "B-dobras", "O", "B-deformação", "O", "B-associadas", "O", "B-corresponde", "O", "B-existente", "O", "B-discordância", "O", "B-datada", "O", "B-givetiano", "O", "B-pedreira", "O", "B-engenharia", "O", "B-cabrela"]}'], ['{"tags": ["O", "B-unidadeEstratigrafica", "B-contextoGeologicoDeBacia", "B-epoca", "B-idade", "B-..."]}'], ['{"tags": ["O", "B-unidadeEstratigrafica", "B-contextoGeologicoDeBacia", "B-epoca", "B-idade", "B-unidadeEstratigrafica", "B-contextoGeologicoDeBaci

c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: época seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: período seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: Bacia do Araripe seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: Conglomerado basal seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\seqeval\metrics\sequence_labe

In [364]:
accuracy_score

<function seqeval.metrics.sequence_labeling.accuracy_score(y_true, y_pred)>

In [365]:
report

{'standard': {'...': {'precision': np.float64(0.0),
   'recall': np.float64(0.0),
   'f1-score': np.float64(0.0),
   'support': np.int64(0)},
  '12': {'precision': np.float64(0.0),
   'recall': np.float64(0.0),
   'f1-score': np.float64(0.0),
   'support': np.int64(0)},
  'B-': {'precision': np.float64(0.0),
   'recall': np.float64(0.0),
   'f1-score': np.float64(0.0),
   'support': np.int64(0)},
  'B-estratigrafia': {'precision': np.float64(0.0),
   'recall': np.float64(0.0),
   'f1-score': np.float64(0.0),
   'support': np.int64(0)},
  'B-subsurface': {'precision': np.float64(0.0),
   'recall': np.float64(0.0),
   'f1-score': np.float64(0.0),
   'support': np.int64(0)},
  'B-sul-americanos': {'precision': np.float64(0.0),
   'recall': np.float64(0.0),
   'f1-score': np.float64(0.0),
   'support': np.int64(0)},
  'Clastos': {'precision': np.float64(0.0),
   'recall': np.float64(0.0),
   'f1-score': np.float64(0.0),
   'support': np.int64(0)},
  'I-estratigrafia': {'precision': np.floa

In [366]:
f1_macro

{'standard': np.float64(0.0),
 'heur_len': np.float64(0.0),
 'heur_rare': np.float64(0.0),
 'adversarial': np.float64(0.000779028765088539)}

In [367]:
precision

{'standard': np.float64(0.0),
 'heur_len': np.float64(0.0),
 'heur_rare': np.float64(0.0),
 'adversarial': np.float64(0.0007715057220007716)}

In [368]:
recall

{'standard': np.float64(0.0),
 'heur_len': np.float64(0.0),
 'heur_rare': np.float64(0.0),
 'adversarial': np.float64(0.0035650623885918)}

In [369]:
metrics_all2

{'standard': {'precision_micro': np.float64(0.0),
  'recall_micro': np.float64(0.0),
  'f1_micro': np.float64(0.0),
  'precision_macro': np.float64(0.0),
  'recall_macro': np.float64(0.0),
  'f1_macro': np.float64(0.0),
  'precision_weighted': np.float64(0.0),
  'recall_weighted': np.float64(0.0),
  'f1_weighted': np.float64(0.0),
  'accuracy': 0.5384177575412635},
 'heur_len': {'precision_micro': np.float64(0.0),
  'recall_micro': np.float64(0.0),
  'f1_micro': np.float64(0.0),
  'precision_macro': np.float64(0.0),
  'recall_macro': np.float64(0.0),
  'f1_macro': np.float64(0.0),
  'precision_weighted': np.float64(0.0),
  'recall_weighted': np.float64(0.0),
  'f1_weighted': np.float64(0.0),
  'accuracy': 0.3188010899182561},
 'heur_rare': {'precision_micro': np.float64(0.0),
  'recall_micro': np.float64(0.0),
  'f1_micro': np.float64(0.0),
  'precision_macro': np.float64(0.0),
  'recall_macro': np.float64(0.0),
  'f1_macro': np.float64(0.0),
  'precision_weighted': np.float64(0.0),
  

In [370]:
precision_micro2 = []
recall_micro2 = []
f1_micro2 = []
f1_macro2 = []
precision_macro2 = []
recall_macro2 = []
f1_weighted2 = []
precision_weighted2 = []
recall_weighted2 = []
accuracy2 = []
name2 = []

for keys,value in metrics_all2.items():
    name2.append(keys)
    precision_micro2.append(value['precision_micro'])
    recall_micro2.append(value['recall_micro'])
    f1_micro2.append(value['f1_micro'])
    precision_macro2.append(value['precision_macro'])
    recall_macro2.append(value['recall_macro'])
    f1_macro2.append(value['f1_macro'])
    precision_weighted2.append(value['precision_weighted'])
    recall_weighted2.append(value['recall_weighted'])
    f1_weighted2.append(value['f1_weighted'])
    accuracy2.append(value['accuracy'])

df_results2 = pd.DataFrame({'name':name2,
                           'precision_micro':precision_micro2,
                           'recall_micro':recall_micro2,
                           'f1_micro':f1_micro2,
                           'precision_macro':precision_macro2,
                           'recall_macro':recall_macro2,
                           'f1_macro':f1_macro2,
                           'precision_weighted':precision_weighted2,
                           'recall_weighted':recall_weighted2,
                           'f1_weighted':f1_weighted2,
                           'accuracy_score':accuracy2})


In [371]:
df_results2.to_csv(f'metricas_{model_escolhido}_complemento.csv', index=False)

In [372]:
df_results2

,name,precision_micro,recall_micro,f1_micro,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,accuracy_score
0,standard,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.538418
1,heur_len,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.318801
2,heur_rare,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.543056
3,adversarial,0.001402,0.009479,0.002442,0.000772,0.003565,0.000779,0.004752,0.009479,0.004765,0.336905
